<a href="https://colab.research.google.com/github/vitorjensen/monitor-trading-margens/blob/main/notebooks/01_analise_exploratoria.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Projeto prático 01** - Monitor de Margens e Trading

### **Objetivo:** Descobrir em qual dia e para qual produto a margem de lucro da mesa de Trading colapsou, tornando inviável a venda daquele combustível/insumo no mercado spot.

##### Configuração do PlayWright para Web Scraper automáticos de dados

In [9]:
# 1. Instala as bibliotecas do Python
!pip install playwright pandas openpyxl

# 2. Baixa os navegadores do Playwright
!playwright install

# 3. Instala as dependências de sistema do Linux para o Chromium rodar na nuvem
!playwright install-deps

Installing dependencies...
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-freefont-ttf is already the newest version (20120503-10build1

In [10]:
import os

# Garante que as pastas do nosso repositório existam no menu lateral do Colab
os.makedirs('src', exist_ok=True)
os.makedirs('data/raw', exist_ok=True)
print("📂 Estrutura de pastas verificada com sucesso!")

📂 Estrutura de pastas verificada com sucesso!


##### Escrevendo o código Web Scraper no caminho: src/scraper.py

In [11]:
%%writefile src/scraper.py
from playwright.sync_api import sync_playwright
import pandas as pd
from datetime import datetime
import os

def executar_scraper():
    print("🚀 [COLAB] Iniciando extração e estruturação de dados...")

    with sync_playwright() as p:
        # Lançando o navegador em modo oculto (obrigatório no Colab)
        browser = p.chromium.launch(headless=True)
        page = browser.new_page()

        # [Simulação do Playwright]
        lista_produtos = ['Etanol Hidratado', 'Lubrificante Hidráulico']
        lista_precos_spot = [3.15, 13.50]

        # Capturando a data do dia de hoje de forma dinâmica
        data_hoje = datetime.now().strftime('%Y-%m-%d')

        # 1. Criando um dicionário estruturado
        dados_mercado = {
            'Data': [data_hoje] * len(lista_produtos),
            'Produto': lista_produtos,
            'Preco_Venda_Spot_Litro': lista_precos_spot
        }

        # 2. Convertendo o dicionário em DataFrame
        df_coletado = pd.DataFrame(dados_mercado)

        # 3. Garantindo que a pasta física de destino exista
        os.makedirs('data/raw', exist_ok=True)

        # 4. Salvando o DataFrame como arquivo CSV na pasta raw
        df_coletado.to_csv('data/raw/precos_mercado.csv', index=False)

        print("✅ Arquivo 'precos_mercado.csv' gerado e salvo com sucesso via Scraper!")

        browser.close()

if __name__ == "__main__":
    executar_scraper()

Overwriting src/scraper.py


In [12]:
!python src/scraper.py

🚀 [COLAB] Iniciando extração e estruturação de dados...
✅ Arquivo 'precos_mercado.csv' gerado e salvo com sucesso via Scraper!


##### Teste do Web Scraper com um site teste

In [13]:
!python src/scraper.py

🚀 [COLAB] Iniciando extração e estruturação de dados...
✅ Arquivo 'precos_mercado.csv' gerado e salvo com sucesso via Scraper!


####Configuração para criar a base de dados de teste utilizando os seeds

In [14]:
import pandas as pd
import numpy as np
import os

In [15]:

# Configuração para criar dados de testes idênticos
np.random.seed(42)
datas = pd.date_range(start='2026-06-01', end='2026-06-15', freq='D')
produtos = ['Etanol Hidratado', 'Lubrificante Hidráulico']

# Gerando combinações de data e produto
lista_comb = [(d, p) for d in datas for p in produtos]
df_base = pd.DataFrame(lista_comb, columns=['Data', 'Produto'])

# --- DATAFRAME 1: CUSTOS INTERNOS DA INDÚSTRIA ---
df_custos = df_base.copy()
df_custos['Custo_Materia_Prima_Litro'] = np.where(df_custos['Produto'] == 'Etanol Hidratado', np.random.uniform(2.10, 2.60, len(df_base)), np.random.uniform(12.00, 15.50, len(df_base)))
df_custos['Custo_Operacional_Litro'] = np.where(df_custos['Produto'] == 'Etanol Hidratado', np.random.uniform(0.40, 0.60, len(df_base)), np.random.uniform(3.00, 4.50, len(df_base)))
df_custos = df_custos.sample(frac=1).reset_index(drop=True)

# --- DATAFRAME 2: PREÇOS DO MERCADO SPOT (TRADING) ---
df_mercado = df_base.copy()
df_mercado['Preco_Venda_Spot_Litro'] = np.where(df_mercado['Produto'] == 'Etanol Hidratado', np.random.uniform(2.80, 3.40, len(df_base)), np.random.uniform(16.00, 21.00, len(df_base)))

# Injetando a anomalia oculta para você caçar na análise!
df_mercado.loc[(df_mercado['Data'] == '2026-06-10') & (df_mercado['Produto'] == 'Lubrificante Hidráulico'), 'Preco_Venda_Spot_Litro'] = 13.50
df_mercado = df_mercado.sample(frac=1).reset_index(drop=True)

# --- EXPORTANDO PARA AS PASTAS DO REPOSITÓRIO ---
# Criando caminhos relativos para funcionar direto na sua pasta local do Mac
os.makedirs('data/raw', exist_ok=True)

df_custos.to_csv('data/raw/custos_industriais.csv', index=False)
df_mercado.to_csv('data/raw/precos_mercado.csv', index=False)

print("✅ Arquivos gerados e salvos com sucesso em:")
print("   -> data/raw/custos_industriais.csv")
print("   -> data/raw/precos_mercado.csv")

✅ Arquivos gerados e salvos com sucesso em:
   -> data/raw/custos_industriais.csv
   -> data/raw/precos_mercado.csv


#### **Passo 1:** Consolidação do **Custo Total** Interno: Saber quanto custa, de fato, produzir cada litro de produto na fábrica.

In [16]:
df_custos_industriais = pd.read_csv('data/raw/custos_industriais.csv')
df_custos_industriais.head()

,Data,Produto,Custo_Materia_Prima_Litro,Custo_Operacional_Litro
0,2026-06-08,Etanol Hidratado,2.190912,0.541371
1,2026-06-01,Lubrificante Hidráulico,12.596834,4.069867
2,2026-06-11,Lubrificante Hidráulico,14.712965,3.241832
3,2026-06-02,Lubrificante Hidráulico,15.321099,3.841916
4,2026-06-06,Lubrificante Hidráulico,13.733119,3.954616


In [17]:
df_custos_industriais['Custo_Total_Litro'] = df_custos_industriais['Custo_Materia_Prima_Litro'] + df_custos_industriais['Custo_Operacional_Litro']
df_custos_industriais.head()

,Data,Produto,Custo_Materia_Prima_Litro,Custo_Operacional_Litro,Custo_Total_Litro
0,2026-06-08,Etanol Hidratado,2.190912,0.541371,2.732284
1,2026-06-01,Lubrificante Hidráulico,12.596834,4.069867,16.666702
2,2026-06-11,Lubrificante Hidráulico,14.712965,3.241832,17.954797
3,2026-06-02,Lubrificante Hidráulico,15.321099,3.841916,19.163015
4,2026-06-06,Lubrificante Hidráulico,13.733119,3.954616,17.687735


#### **Passo 2:** Mesclar as planilhas para sabermos o preço de mercado e o custo juntos

In [18]:
df_precos_mercado = pd.read_csv('data/raw/precos_mercado.csv')
df_precos_mercado.head()

,Data,Produto,Preco_Venda_Spot_Litro
0,2026-06-05,Lubrificante Hidráulico,20.683650
1,2026-06-13,Lubrificante Hidráulico,17.695149
2,2026-06-03,Etanol Hidratado,2.980527
3,2026-06-06,Lubrificante Hidráulico,17.705332
4,2026-06-04,Etanol Hidratado,2.822132


In [19]:
df_merge = pd.merge(df_custos_industriais, df_precos_mercado, on=['Data', 'Produto'])
df_merge.head()

,Data,Produto,Custo_Materia_Prima_Litro,Custo_Operacional_Litro,Custo_Total_Litro,Preco_Venda_Spot_Litro
0,2026-06-08,Etanol Hidratado,2.190912,0.541371,2.732284,3.093672
1,2026-06-01,Lubrificante Hidráulico,12.596834,4.069867,16.666702,19.387822
2,2026-06-11,Lubrificante Hidráulico,14.712965,3.241832,17.954797,16.465514
3,2026-06-02,Lubrificante Hidráulico,15.321099,3.841916,19.163015,18.560465
4,2026-06-06,Lubrificante Hidráulico,13.733119,3.954616,17.687735,17.705332


##### Definir quais produtos estão com um Custo Total MAIOR que um Preço de Venda

In [20]:
df_custo_maior = df_merge[df_merge['Custo_Total_Litro'] > df_merge['Preco_Venda_Spot_Litro']]
df_custo_maior

,Data,Produto,Custo_Materia_Prima_Litro,Custo_Operacional_Litro,Custo_Total_Litro,Preco_Venda_Spot_Litro
2,2026-06-11,Lubrificante Hidráulico,14.712965,3.241832,17.954797,16.465514
3,2026-06-02,Lubrificante Hidráulico,15.321099,3.841916,19.163015,18.560465
6,2026-06-08,Lubrificante Hidráulico,14.318828,3.373938,17.692766,17.289708
10,2026-06-15,Etanol Hidratado,2.396207,0.577443,2.973650,2.911911
19,2026-06-02,Etanol Hidratado,2.465997,0.565748,3.031744,2.951069
22,2026-06-10,Lubrificante Hidráulico,12.646991,3.115470,15.762460,13.500000
29,2026-06-13,Lubrificante Hidráulico,15.226560,4.307191,19.533751,17.695149


#### **Passo 3:** Calcular a Margem de Arbitragem: Descobrir a sobra financeira de cada operação de Trading

In [21]:
df_merge['Margem_Absoluta_Litro'] = df_merge['Preco_Venda_Spot_Litro'] - df_merge['Custo_Total_Litro']
df_merge.head()

,Data,Produto,Custo_Materia_Prima_Litro,Custo_Operacional_Litro,Custo_Total_Litro,Preco_Venda_Spot_Litro,Margem_Absoluta_Litro
0,2026-06-08,Etanol Hidratado,2.190912,0.541371,2.732284,3.093672,0.361388
1,2026-06-01,Lubrificante Hidráulico,12.596834,4.069867,16.666702,19.387822,2.721120
2,2026-06-11,Lubrificante Hidráulico,14.712965,3.241832,17.954797,16.465514,-1.489283
3,2026-06-02,Lubrificante Hidráulico,15.321099,3.841916,19.163015,18.560465,-0.602550
4,2026-06-06,Lubrificante Hidráulico,13.733119,3.954616,17.687735,17.705332,0.017597


In [22]:
df_merge['Margem_Percentual'] = (df_merge['Margem_Absoluta_Litro'] / df_merge['Preco_Venda_Spot_Litro']) * 100
df_merge.head()

,Data,Produto,Custo_Materia_Prima_Litro,Custo_Operacional_Litro,Custo_Total_Litro,Preco_Venda_Spot_Litro,Margem_Absoluta_Litro,Margem_Percentual
0,2026-06-08,Etanol Hidratado,2.190912,0.541371,2.732284,3.093672,0.361388,11.681515
1,2026-06-01,Lubrificante Hidráulico,12.596834,4.069867,16.666702,19.387822,2.721120,14.035203
2,2026-06-11,Lubrificante Hidráulico,14.712965,3.241832,17.954797,16.465514,-1.489283,-9.044862
3,2026-06-02,Lubrificante Hidráulico,15.321099,3.841916,19.163015,18.560465,-0.602550,-3.246416
4,2026-06-06,Lubrificante Hidráulico,13.733119,3.954616,17.687735,17.705332,0.017597,0.099388


#### **Passo 4:** Diagnóstico Final: Onde está a Crise?

##### Achando os produtos que ficaram abaixo do nível aceitável

In [23]:
df_produtos_abaixo_do_nivel = df_merge[df_merge['Margem_Percentual'] < 0].sort_values(by='Margem_Percentual', ascending=True)
df_produtos_abaixo_do_nivel

,Data,Produto,Custo_Materia_Prima_Litro,Custo_Operacional_Litro,Custo_Total_Litro,Preco_Venda_Spot_Litro,Margem_Absoluta_Litro,Margem_Percentual
22,2026-06-10,Lubrificante Hidráulico,12.646991,3.115470,15.762460,13.500000,-2.262460,-16.758966
29,2026-06-13,Lubrificante Hidráulico,15.226560,4.307191,19.533751,17.695149,-1.838602,-10.390428
2,2026-06-11,Lubrificante Hidráulico,14.712965,3.241832,17.954797,16.465514,-1.489283,-9.044862
3,2026-06-02,Lubrificante Hidráulico,15.321099,3.841916,19.163015,18.560465,-0.602550,-3.246416
19,2026-06-02,Etanol Hidratado,2.465997,0.565748,3.031744,2.951069,-0.080675,-2.733758
6,2026-06-08,Lubrificante Hidráulico,14.318828,3.373938,17.692766,17.289708,-0.403058,-2.331203
10,2026-06-15,Etanol Hidratado,2.396207,0.577443,2.973650,2.911911,-0.061739,-2.120213


##### Calculando os produtos que tiveram pior performance

In [24]:
df_produtos_pior_performance = df_merge.groupby('Produto')['Margem_Percentual'].mean().reset_index()
df_produtos_pior_performance

,Produto,Margem_Percentual
0,Etanol Hidratado,9.043627
1,Lubrificante Hidráulico,4.866700


#### Criando o próximo arquivo "Pipeline" para continuação do projeto

In [25]:
%%writefile src/pipeline.py

Overwriting src/pipeline.py


In [26]:
import pandas as pd
import os

def rodar_pipeline_analise():
    print("🧠 [PIPELINE] Iniciando processamento de margens e governança...")

    # 1. Caminhos dos arquivos
    caminho_custos = 'data/raw/custos_industriais.csv'
    caminho_mercado = 'data/raw/precos_mercado.csv'

    # Verificação de segurança: os arquivos de entrada existem?
    if not os.path.exists(caminho_custos) or not os.path.exists(caminho_mercado):
        print("❌ Erro: Arquivos brutos (raw) não encontrados. Execute o gerador e o scraper primeiro.")
        return

    # 2. Carga dos dados brutos
    df_custos = pd.read_csv(caminho_custos)
    df_mercado = pd.read_csv(caminho_mercado)

    # 3. Passo 1 & 2: Cálculo do Custo Total e Cruzamento de Dados (Merge)
    # Nota: Caso os custos industriais tenham múltiplas datas, o merge alinhará perfeitamente.
    df_custos['Custo_Total_Litro'] = df_custos['Custo_Materia_Prima_Litro'] + df_custos['Custo_Operacional_Litro']

    df_merge = pd.merge(
        df_mercado,
        df_custos[['Data', 'Produto', 'Custo_Total_Litro']],
        on=['Data', 'Produto'],
        how='inner'
    )

    # 4. Passo 3: Cálculo das Margens (Lógica corrigida e validada por você!)
    df_merge['Margem_Absoluta_Litro'] = df_merge['Preco_Venda_Spot_Litro'] - df_merge['Custo_Total_Litro']
    df_merge['Margem_Percentual'] = (df_merge['Margem_Absoluta_Litro'] / df_merge['Preco_Venda_Spot_Litro']) * 100

    # 5. Sistema de Governança (Alerta de Margem Negativa)
    Prejuizos = df_merge[df_merge['Margem_Absoluta_Litro'] < 0]

    print("\n------------------------------------------------------------")
    if not Prejuizos.empty:
        print("🚨🚨 ALERTAS DE GOVERNANÇA DETECTADOS (MARGEM NEGATIVA)! 🚨🚨")
        for idx, linha in Prejuizos.iterrows():
            print(f"⚠️ PRODUTO: {linha['Produto']} | DATA: {linha['Data']}")
            print(f"   Preço Spot: R$ {linha['Preco_Venda_Spot_Litro']:.2f} | Custo Total: R$ {linha['Custo_Total_Litro']:.2f}")
            print(f"   Margem: {linha['Margem_Percentual']:.2f}% (Prejuízo de R$ {abs(linha['Margem_Absoluta_Litro']):.2f}/L)")
    else:
        print("✅ Sucesso: Todas as operações da rodada operando com margem positiva.")
    print("------------------------------------------------------------\n")

    # 6. Salvando o resultado final na pasta de dados processados (Camada Silver/Trusted)
    os.makedirs('data/processed', exist_ok=True)
    df_merge.to_csv('data/processed/relatorio_margens.csv', index=False)
    print("💾 Dados consolidados salvos com sucesso em: data/processed/relatorio_margens.csv")

if __name__ == "__main__":
    rodar_pipeline_analise()

🧠 [PIPELINE] Iniciando processamento de margens e governança...

------------------------------------------------------------
🚨🚨 ALERTAS DE GOVERNANÇA DETECTADOS (MARGEM NEGATIVA)! 🚨🚨
⚠️ PRODUTO: Lubrificante Hidráulico | DATA: 2026-06-13
   Preço Spot: R$ 17.70 | Custo Total: R$ 19.53
   Margem: -10.39% (Prejuízo de R$ 1.84/L)
⚠️ PRODUTO: Etanol Hidratado | DATA: 2026-06-15
   Preço Spot: R$ 2.91 | Custo Total: R$ 2.97
   Margem: -2.12% (Prejuízo de R$ 0.06/L)
⚠️ PRODUTO: Lubrificante Hidráulico | DATA: 2026-06-11
   Preço Spot: R$ 16.47 | Custo Total: R$ 17.95
   Margem: -9.04% (Prejuízo de R$ 1.49/L)
⚠️ PRODUTO: Lubrificante Hidráulico | DATA: 2026-06-08
   Preço Spot: R$ 17.29 | Custo Total: R$ 17.69
   Margem: -2.33% (Prejuízo de R$ 0.40/L)
⚠️ PRODUTO: Etanol Hidratado | DATA: 2026-06-02
   Preço Spot: R$ 2.95 | Custo Total: R$ 3.03
   Margem: -2.73% (Prejuízo de R$ 0.08/L)
⚠️ PRODUTO: Lubrificante Hidráulico | DATA: 2026-06-02
   Preço Spot: R$ 18.56 | Custo Total: R$ 19.16
   Ma